In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import linregress

# Варіант 1 (Київ)
np.random.seed(1)
base_temp = 9.5
amplitude = 14.0
trend_per_year = 0.04
noise_scale = 1.0
city = "Київ"

n_years = 10
dates = pd.date_range(start="2015-01-01", periods=n_years * 12, freq="MS")
month = dates.month
years_elapsed = (dates.year - dates.year[0]) + (dates.month - 1) / 12

seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
trend = trend_per_year * years_elapsed
noise = np.random.normal(0, noise_scale, len(dates))
temp = np.round(base_temp + trend + seasonal + noise, 1)

climate_ts = pd.Series(temp, index=dates, name="температура")
print(climate_ts.head(12))
print(f"Загальна кількість місяців: {len(climate_ts)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index, climate_ts.values, color="tab:blue", lw=1.2)
ax.set_xlabel("Час (Місяць / Рік)")
ax.set_ylabel("Середньомісячна температура, °C")
ax.set_title(f"Багаторічний часовий ряд температур для міста {city} (2015–2024)")
ax.grid(True)
plt.show()

In [ ]:
ma3 = climate_ts.rolling(window=3, center=True).mean()
ma12_centered = climate_ts.rolling(window=12, center=True).mean()
ma12_trailing = climate_ts.rolling(window=12, center=False).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index, climate_ts.values, alpha=0.3, label="Оригінал", color="gray")
ax.plot(climate_ts.index, ma3.values, label="Вікно = 3 (центроване)", color="tab:orange", lw=1.5)
ax.plot(climate_ts.index, ma12_centered.values, label="Вікно = 12 (центроване)", color="tab:red", lw=2)
ax.plot(climate_ts.index, ma12_trailing.values, label="Вікно = 12 (трейлінгове)", color="tab:green", ls="--", lw=1.5)
ax.legend()
ax.set_title("Ковзне середнє: порівняння розмірів вікон і типів центрування")
ax.grid(True)
plt.show()

In [ ]:
decomposition = seasonal_decompose(climate_ts, model="additive", period=12)
fig = decomposition.plot()
fig.set_size_inches(10, 7)
plt.tight_layout()
plt.show()

print("Перші 12 значень сезонної складової:")
print(decomposition.seasonal.head(12).round(2))
print(f"\nСтандартне відхилення залишку (resid): {decomposition.resid.dropna().std():.3f}")

In [ ]:
t = np.arange(len(climate_ts))
known = decomposition.trend.notna().values
fit = linregress(t[known], decomposition.trend[known])

n_ahead = 12
t_future = np.arange(len(climate_ts), len(climate_ts) + n_ahead)
trend_forecast = fit.intercept + fit.slope * t_future

seasonal_by_month = decomposition.seasonal.groupby(decomposition.seasonal.index.month).mean()
future_dates = pd.date_range(climate_ts.index[-1] + pd.offsets.MonthBegin(1), periods=n_ahead, freq="MS")
seasonal_forecast = seasonal_by_month.reindex(future_dates.month).values

forecast = pd.Series(trend_forecast + seasonal_forecast, index=future_dates)
print("Прогноз на 12 місяців уперед:")
print(forecast.round(1))

# Графік історичних даних та прогнозу
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index[-36:], climate_ts.values[-36:], label="Історія (останні 3 роки)", color="tab:blue")
ax.plot(forecast.index, forecast.values, label="Прогноз (на рік уперед)", color="tab:orange", marker="o")
ax.legend()
ax.set_title("Прогноз температури на 12 місяців уперед")
ax.grid(True)
plt.show()

In [ ]:
deseason = climate_ts - decomposition.seasonal
t_years = np.arange(len(climate_ts)) / 12
trend_test = linregress(t_years, deseason.values)

print(f"Нахил тренду: {trend_test.slope:.4f} °C/рік")
print(f"Стандартна похибка (stderr): {trend_test.stderr:.4f}")
print(f"p-value: {trend_test.pvalue:.4e}")

ci_lower = trend_test.slope - 1.96 * trend_test.stderr
ci_upper = trend_test.slope + 1.96 * trend_test.stderr
print(f"95% довірчий інтервал для нахилу: [{ci_lower:.4f}, {ci_upper:.4f}]")

## Контрольні питання

1. **Чому часовий ряд не можна аналізувати звичайними методами незалежних вибірок?**
   Елементи часового ряду порушують ключову умову незалежності спостережень через наявність автокореляції та часової структури (сусідні місяці тісно пов'язані між собою).

2. **Чому замале й завелике вікно ковзного середнього є проблемою?**
   Замале вікно не згладжує випадковий шум, а завелике вікно (не кратне періоду сезонності) спотворює структуру тренду та призводить до згладжування важливих локальних особливостей часового ряду.

3. **У чому різниця між трейлінговим і центрованим ковзним середнім?**
   Трейлінгове вікно розраховується за попередніми періодами (`center=False`), створюючи систематичне запізнення (лаг), тоді як центроване вікно (`center=True`) симетрично охоплює точки довкола поточного періоду, але вимагає «заглядання в майбутнє».

4. **Чому саме p-value нахилу визначає наявність тренду?**
   Сам по собі знак або величина коефіцієнта нахилу можуть бути наслідком випадкового збігу (шуму). Статистичний тест перевіряє, чи відрізняється нахил від нуля суттєво більше, ніж це може статися випадково через похибку вимірювань.